# Notebook 01 — Mesocorticolimbic Circuit Demo

Demonstrates the ODE firing-rate model of the dopaminergic circuit under
baseline conditions and under simulated D2 antagonism (haloperidol-like).

**Circuit nodes**: VTA · NAcD1 (direct) · NAcD2 (indirect) · PFC  
**Model**: Leaky integrator with sigmoid transfer function  
**References**: Humphries & Prescott 2010; Frank 2005


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from neuropharm_sim.circuit_model import CircuitParams, MesocorticolimbicCircuit
from neuropharm_sim.visualization import plot_circuit_dynamics, plot_occupancy_curves

## 1. Receptor binding curves

D1 (Kd = 1 500 nM) vs D2 (Kd = 20 nM): the 75-fold affinity difference
is the mechanistic basis for tonic DA preferentially activating D2.

In [ ]:
fig, axes = plot_occupancy_curves()
plt.show()

## 2. Baseline circuit dynamics

No drug. System settles to physiological tonic state.

In [ ]:
baseline_circuit = MesocorticolimbicCircuit()
traj_baseline = baseline_circuit.simulate(t_span_ms=(0, 1000), n_points=500)

fig, axes = plot_circuit_dynamics(traj_baseline, title="Baseline Circuit (no drug)")
plt.show()

ss = baseline_circuit.steady_state()
print(f"Steady-state firing rates:")
print(f"  VTA   = {ss.r_vta:.3f}")
print(f"  NAcD1 = {ss.r_nacd1:.3f}")
print(f"  NAcD2 = {ss.r_nacd2:.3f}")
print(f"  PFC   = {ss.r_pfc:.3f}")
print(f"  DA    = {ss.da_nm:.1f} nM")
print(f"  D1 occ = {ss.d1_occ*100:.1f}%")
print(f"  D2 occ = {ss.d2_occ*100:.1f}%")

## 3. D2 antagonism (haloperidol-equivalent)

At 3.6 nM free brain concentration (Ki_D2 = 1.2 nM → ~75% D2 occupancy),
D2-MSN disinhibition shifts the indirect pathway.

In [ ]:
haldol_circuit = MesocorticolimbicCircuit(
    drug_d2_antagonist_nm=3.6,
    drug_ki_d2_nm=1.2,
)
traj_haldol = haldol_circuit.simulate(t_span_ms=(0, 1000), n_points=500)

fig, axes = plot_circuit_dynamics(traj_haldol,
    title="Haloperidol (~75% D2 block)")
plt.show()

## 4. Side-by-side comparison: key circuit changes

In [ ]:
labels = ["VTA", "NAcD1", "NAcD2", "PFC"]
keys   = ["r_vta", "r_nacd1", "r_nacd2", "r_pfc"]
colors = ["#b35806", "#2166ac", "#d6604d", "#4dac26"]

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
axes = axes.flatten()

t = traj_baseline["t_ms"]
for ax, label, key, color in zip(axes, labels, keys, colors):
    ax.plot(t, traj_baseline[key], lw=1.8, color=color, label="Baseline", alpha=0.9)
    ax.plot(t, traj_haldol[key],   lw=1.8, color=color, label="Haloperidol",
            ls="--", alpha=0.9)
    ax.set_title(label, fontweight="bold")
    ax.set_ylabel("Firing rate (norm.)")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9)

for ax in axes[-2:]:
    ax.set_xlabel("Time (ms)")

fig.suptitle("Circuit Changes Under D2 Antagonism", fontweight="bold", fontsize=13)
fig.tight_layout()
plt.show()

## 5. DA concentration — cocaine vs baseline

In [ ]:
cocaine_circuit = MesocorticolimbicCircuit(drug_da_multiplier=1.78)  # DAT block 78%
traj_cocaine = cocaine_circuit.simulate(t_span_ms=(0, 1000), n_points=500)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(traj_baseline["t_ms"], traj_baseline["da_nm"],
        lw=2, label="Baseline", color="#aaaaaa")
ax.plot(traj_cocaine["t_ms"], traj_cocaine["da_nm"],
        lw=2, label="Cocaine (78% DAT block)", color="#d6604d")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Extracellular DA (nM)")
ax.set_title("Cocaine elevates synaptic dopamine", fontweight="bold")
ax.axhline(100, ls="--", color="gray", alpha=0.5, label="Tonic baseline (100 nM)")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()